# Metodología  
1. hacer reducción dimensional con regularización lasso o con Randon Forest
2. 

# Etapa de reducción dimensional  

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\1_raw\1_meteo_epi_2021-2026_1_rezagos_sin_nulos.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados"
output_file = output_dir + r"\datos_reducidos_lasso.xlsx"

# Cargar datos
print("Cargando datos...")
df = pd.read_excel(input_file)
print(f"Dimensiones originales: {df.shape}")

# Identificar columnas
columns_to_exclude = ['fecha', 'año', 'semana_epi', 'casos_dengue']
predictor_cols = [col for col in df.columns if col not in columns_to_exclude]
target_col = 'casos_dengue'

print(f"Variables predictoras: {len(predictor_cols)}")
print(f"Variable objetivo: {target_col}")

# Preparar datos
X = df[predictor_cols].values
y = df[target_col].values

# Asegurar que y sea 1D
if len(y.shape) > 1:
    y = y.ravel()

# Manejar valores infinitos y NaN
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

print("\nAplicando Lasso para selección de características...")

# Probar diferentes valores de alpha
alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
best_alpha = None
best_n_features = 0
best_coef = None

# Buscar alpha que nos dé <= 19 features (para tener menos de 20)
for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000, random_state=42)
    lasso.fit(X, y)
    
    n_selected = np.sum(np.abs(lasso.coef_) > 1e-6)
    print(f"Alpha={alpha}: {n_selected} features seleccionados")
    
    if n_selected <= 19 and n_selected > best_n_features:
        best_n_features = n_selected
        best_alpha = alpha
        best_coef = lasso.coef_

# Si ningún alpha dio <= 19, usar el más restrictivo
if best_alpha is None:
    print("\nNingún alpha dio <= 19 features. Usando alpha=10.0")
    best_alpha = 10.0
    lasso = Lasso(alpha=best_alpha, max_iter=10000, random_state=42)
    lasso.fit(X, y)
    best_coef = lasso.coef_
    best_n_features = np.sum(np.abs(best_coef) > 1e-6)

print(f"\nAlpha seleccionado: {best_alpha}")
print(f"Features seleccionados: {best_n_features}")

# Seleccionar features importantes
selected_mask = np.abs(best_coef) > 1e-6
selected_features = [predictor_cols[i] for i in range(len(predictor_cols)) if selected_mask[i]]
selected_coefs = best_coef[selected_mask]

# Crear dataframe con resultados
print("\nFeatures seleccionados y sus coeficientes:")
selected_df = pd.DataFrame({
    'Feature': selected_features,
    'Coeficiente': selected_coefs,
    'Abs_Coeficiente': np.abs(selected_coefs)
})
selected_df = selected_df.sort_values('Abs_Coeficiente', ascending=False)
print(selected_df)

# Crear dataset reducido
print(f"\nCreando dataset reducido con {len(selected_features)} predictores...")
X_reduced = X[:, selected_mask]
df_reduced = pd.DataFrame(X_reduced, columns=selected_features)

# Añadir columnas no predictoras
for col in columns_to_exclude:
    if col in df.columns:
        df_reduced[col] = df[col].values

# Reordenar columnas para que fecha, año, semana_epi y casos_dengue estén al inicio
cols_order = [col for col in columns_to_exclude if col in df_reduced.columns] + \
             [col for col in df_reduced.columns if col not in columns_to_exclude]
df_reduced = df_reduced[cols_order]

# Guardar dataset reducido
print(f"Guardando dataset reducido en: {output_file}")
df_reduced.to_excel(output_file, index=False)

# Resumen final
print("\n" + "="*50)
print("RESUMEN FINAL")
print("="*50)
print(f"Dimensiones originales: {df.shape}")
print(f"Dimensiones reducidas: {df_reduced.shape}")
print(f"Número de predictores originales: {len(predictor_cols)}")
print(f"Número de predictores seleccionados: {len(selected_features)}")
print(f"Variables excluidas (fecha, año, semana_epi, casos_dengue): {len([col for col in columns_to_exclude if col in df.columns])}")
print(f"\nTop 10 features más importantes:")
print(selected_df.head(10))
print("="*50)

Cargando datos...
Dimensiones originales: (270, 172)
Variables predictoras: 168
Variable objetivo: casos_dengue

Aplicando Lasso para selección de características...
Alpha=0.001: 159 features seleccionados
Alpha=0.005: 136 features seleccionados
Alpha=0.01: 122 features seleccionados
Alpha=0.05: 92 features seleccionados
Alpha=0.1: 83 features seleccionados
Alpha=0.5: 52 features seleccionados
Alpha=1.0: 37 features seleccionados
Alpha=5.0: 26 features seleccionados
Alpha=10.0: 19 features seleccionados

Alpha seleccionado: 10.0
Features seleccionados: 19

Features seleccionados y sus coeficientes:
               Feature  Coeficiente  Abs_Coeficiente
15  casos_dengue_lag_1     0.544864         0.544864
16  casos_dengue_lag_2     0.285243         0.285243
17  casos_dengue_lag_4     0.053882         0.053882
18  casos_dengue_lag_8     0.040782         0.040782
2           prec_lag_4     0.018776         0.018776
3          prec_lag_10     0.009545         0.009545
6            soi_lag_3 

# 2. Entrenar el modelo  



In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.model_selection import TimeSeriesSplit
import optuna

# Configuración de rutas
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados\datos_reducidos_lasso.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
os.makedirs(output_dir, exist_ok=True)

# Cargar datos
print("Cargando datos reducidos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])

# Identificar columnas
target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']
predictor_cols = [col for col in df.columns if col not in exclude_cols + [target_col]]

print(f"Predictores: {len(predictor_cols)}")
print(f"Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")

# Función para entrenar y evaluar modelo
def train_evaluate_model(X_train, y_train, X_test, y_test, train_dates, test_dates, 
                         train_label, model_params=None):
    """Entrena modelo LightGBM y retorna métricas y predicciones"""
    
    if model_params is None:
        model_params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.9,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42,
            'min_child_samples': 20,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1
        }
    
    # Crear datasets
    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_test = lgb.Dataset(X_test, y_test, reference=lgb_train)
    
    # Entrenar
    model = lgb.train(
        model_params,
        lgb_train,
        valid_sets=[lgb_train, lgb_test],
        valid_names=['train', 'test'],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    # Predicciones
    y_train_pred = model.predict(X_train, num_iteration=model.best_iteration)
    y_test_pred = model.predict(X_test, num_iteration=model.best_iteration)
    
    # Métricas
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # MAPE (evitando división por cero)
    train_mape = np.mean(np.abs((y_train - y_train_pred) / (y_train + 1))) * 100
    test_mape = np.mean(np.abs((y_test - y_test_pred) / (y_test + 1))) * 100
    
    # Error en picos (top 10% de valores)
    threshold_train = np.percentile(y_train, 90)
    threshold_test = np.percentile(y_test, 90)
    
    peak_mask_train = y_train >= threshold_train
    peak_mask_test = y_test >= threshold_test
    
    peak_mae_train = mean_absolute_error(y_train[peak_mask_train], y_train_pred[peak_mask_train]) if np.sum(peak_mask_train) > 0 else np.nan
    peak_mae_test = mean_absolute_error(y_test[peak_mask_test], y_test_pred[peak_mask_test]) if np.sum(peak_mask_test) > 0 else np.nan
    
    metrics = {
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mape': train_mape,
        'test_mape': test_mape,
        'peak_mae_train': peak_mae_train,
        'peak_mae_test': peak_mae_test,
        'best_iteration': model.best_iteration,
        'feature_importance': model.feature_importance(importance_type='gain')
    }
    
    return model, metrics, y_train_pred, y_test_pred

# Función para optimizar hiperparámetros
def optimize_lightgbm(X_train, y_train, X_val, y_val):
    """Optimiza hiperparámetros usando Optuna"""
    
    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'num_leaves': trial.suggest_int('num_leaves', 10, 100),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42
        }
        
        # Entrenar con early stopping
        dtrain = lgb.Dataset(X_train, y_train)
        dval = lgb.Dataset(X_val, y_val, reference=dtrain)
        
        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dtrain, dval],
            valid_names=['train', 'val'],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        
        # Predecir y calcular MAE en validación
        y_pred = model.predict(X_val, num_iteration=model.best_iteration)
        mae = mean_absolute_error(y_val, y_pred)
        
        return mae
    
    # Crear estudio Optuna
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    print(f"\nMejores parámetros encontrados:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    print(f"Mejor MAE en validación: {study.best_value:.4f}")
    
    return study.best_params

# Preparar datos para ambos periodos
print("\nPreparando datos...")

# Periodo 1: 2021-2025
train_mask_1 = df['año'].isin([2021, 2022, 2023, 2024, 2025])
test_mask = df['año'] == 2026

X_train_1 = df[train_mask_1][predictor_cols].values
y_train_1 = df[train_mask_1][target_col].values
train_dates_1 = df[train_mask_1]['fecha'].values

# Periodo 2: 2022-2025
train_mask_2 = df['año'].isin([2022, 2023, 2024, 2025])

X_train_2 = df[train_mask_2][predictor_cols].values
y_train_2 = df[train_mask_2][target_col].values
train_dates_2 = df[train_mask_2]['fecha'].values

# Test (ambos periodos usan el mismo test)
X_test = df[test_mask][predictor_cols].values
y_test = df[test_mask][target_col].values
test_dates = df[test_mask]['fecha'].values

print(f"Periodo 1 - Entrenamiento 2021-2025: {len(X_train_1)} registros")
print(f"Periodo 2 - Entrenamiento 2022-2025: {len(X_train_2)} registros")
print(f"Test 2026: {len(X_test)} registros")

# Optimizar hiperparámetros con validación temporal
print("\nOptimizando hiperparámetros con validación temporal...")

# Usar datos de 2021-2024 para optimización y 2025 para validación
opt_train_mask = df['año'].isin([2021, 2022, 2023, 2024])
opt_val_mask = df['año'] == 2025

X_opt_train = df[opt_train_mask][predictor_cols].values
y_opt_train = df[opt_train_mask][target_col].values
X_opt_val = df[opt_val_mask][predictor_cols].values
y_opt_val = df[opt_val_mask][target_col].values

best_params = optimize_lightgbm(X_opt_train, y_opt_train, X_opt_val, y_opt_val)

# Parámetros base mejorados
base_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42
}

# Combinar parámetros optimizados con base
model_params = {**base_params, **best_params}

print("\nEntrenando modelos con los mejores parámetros...")

# Entrenar modelo Periodo 1 (2021-2025)
model_1, metrics_1, y_train_pred_1, y_test_pred_1 = train_evaluate_model(
    X_train_1, y_train_1, X_test, y_test, 
    train_dates_1, test_dates, 
    "2021-2025", model_params
)

# Entrenar modelo Periodo 2 (2022-2025)
model_2, metrics_2, y_train_pred_2, y_test_pred_2 = train_evaluate_model(
    X_train_2, y_train_2, X_test, y_test,
    train_dates_2, test_dates,
    "2022-2025", model_params
)

# Crear visualizaciones mejoradas
print("\nGenerando gráficos...")

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('Comparativa Modelos LightGBM con Optimización - Reducción Lasso', fontsize=16, fontweight='bold')

# 1. Series temporales - Periodo 1 (2021-2025)
ax1 = axes[0, 0]
ax1.plot(train_dates_1, y_train_pred_1, 'r-', label='Predicho', linewidth=1.5, alpha=0.8)
ax1.plot(train_dates_1, y_train_1, 'b-', label='Real', linewidth=1.5, alpha=0.8)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos Dengue')
ax1.set_title(f'Entrenamiento 2021-2025\nMAE: {metrics_1["train_mae"]:.2f}')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. Series temporales - Periodo 2 (2022-2025)
ax2 = axes[0, 1]
ax2.plot(train_dates_2, y_train_pred_2, 'r-', label='Predicho', linewidth=1.5, alpha=0.8)
ax2.plot(train_dates_2, y_train_2, 'b-', label='Real', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos Dengue')
ax2.set_title(f'Entrenamiento 2022-2025\nMAE: {metrics_2["train_mae"]:.2f}')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Test 2026 - Ambos modelos comparados
ax3 = axes[0, 2]
ax3.plot(test_dates, y_test, 'k-', label='Real', linewidth=2, marker='o', markersize=6)
ax3.plot(test_dates, y_test_pred_1, 'r-', label=f'Modelo 2021-2025 (MAE: {metrics_1["test_mae"]:.2f})', 
         linewidth=1.5, marker='s', markersize=5, alpha=0.8)
ax3.plot(test_dates, y_test_pred_2, 'g-', label=f'Modelo 2022-2025 (MAE: {metrics_2["test_mae"]:.2f})', 
         linewidth=1.5, marker='^', markersize=5, alpha=0.8)
ax3.set_xlabel('Fecha')
ax3.set_ylabel('Casos Dengue')
ax3.set_title('Predicciones en Test (2026)')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# 4. Scatter - Modelo 1
ax4 = axes[1, 0]
ax4.scatter(y_test, y_test_pred_1, alpha=0.6, color='red', s=50)
ax4.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
ax4.set_xlabel('Valores Reales')
ax4.set_ylabel('Predicciones')
ax4.set_title(f'Modelo 2021-2025 - Test\nR²: {metrics_1["test_r2"]:.4f}')
ax4.grid(True, alpha=0.3)

# 5. Scatter - Modelo 2
ax5 = axes[1, 1]
ax5.scatter(y_test, y_test_pred_2, alpha=0.6, color='green', s=50)
ax5.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
ax5.set_xlabel('Valores Reales')
ax5.set_ylabel('Predicciones')
ax5.set_title(f'Modelo 2022-2025 - Test\nR²: {metrics_2["test_r2"]:.4f}')
ax5.grid(True, alpha=0.3)

# 6. Comparación de errores
ax6 = axes[1, 2]
models = ['2021-2025', '2022-2025']
train_mae = [metrics_1['train_mae'], metrics_2['train_mae']]
test_mae = [metrics_1['test_mae'], metrics_2['test_mae']]
peak_mae = [metrics_1['peak_mae_test'], metrics_2['peak_mae_test']]

x = np.arange(len(models))
width = 0.25

bars1 = ax6.bar(x - width, train_mae, width, label='Entrenamiento', color='skyblue')
bars2 = ax6.bar(x, test_mae, width, label='Test', color='lightcoral')
bars3 = ax6.bar(x + width, peak_mae, width, label='Picos (Test)', color='gold')

ax6.set_xlabel('Modelo')
ax6.set_ylabel('MAE')
ax6.set_title('Comparación de Errores')
ax6.set_xticks(x)
ax6.set_xticklabels(models)
ax6.legend()
ax6.grid(True, alpha=0.3)

# 7. Importancia de características - Modelo 1
ax7 = axes[2, 0]
importance_1 = metrics_1['feature_importance']
feature_importance_df_1 = pd.DataFrame({
    'Feature': predictor_cols,
    'Importance': importance_1
}).sort_values('Importance', ascending=True)

top_features_1 = feature_importance_df_1.tail(10)
ax7.barh(top_features_1['Feature'], top_features_1['Importance'], color='red', alpha=0.7)
ax7.set_xlabel('Importancia')
ax7.set_title('Top 10 Features - Modelo 2021-2025')
ax7.grid(True, alpha=0.3)

# 8. Importancia de características - Modelo 2
ax8 = axes[2, 1]
importance_2 = metrics_2['feature_importance']
feature_importance_df_2 = pd.DataFrame({
    'Feature': predictor_cols,
    'Importance': importance_2
}).sort_values('Importance', ascending=True)

top_features_2 = feature_importance_df_2.tail(10)
ax8.barh(top_features_2['Feature'], top_features_2['Importance'], color='green', alpha=0.7)
ax8.set_xlabel('Importancia')
ax8.set_title('Top 10 Features - Modelo 2022-2025')
ax8.grid(True, alpha=0.3)

# 9. Resumen de métricas
ax9 = axes[2, 2]
ax9.axis('off')

# Crear tabla de métricas
metrics_summary = pd.DataFrame({
    'Métrica': ['MAE Train', 'MAE Test', 'RMSE Train', 'RMSE Test', 'R² Train', 'R² Test', 'MAE Picos'],
    'Modelo 2021-2025': [
        f"{metrics_1['train_mae']:.2f}",
        f"{metrics_1['test_mae']:.2f}",
        f"{metrics_1['train_rmse']:.2f}",
        f"{metrics_1['test_rmse']:.2f}",
        f"{metrics_1['train_r2']:.4f}",
        f"{metrics_1['test_r2']:.4f}",
        f"{metrics_1['peak_mae_test']:.2f}"
    ],
    'Modelo 2022-2025': [
        f"{metrics_2['train_mae']:.2f}",
        f"{metrics_2['test_mae']:.2f}",
        f"{metrics_2['train_rmse']:.2f}",
        f"{metrics_2['test_rmse']:.2f}",
        f"{metrics_2['train_r2']:.4f}",
        f"{metrics_2['test_r2']:.4f}",
        f"{metrics_2['peak_mae_test']:.2f}"
    ]
})

# Crear tabla en el gráfico
table = ax9.table(cellText=metrics_summary.values,
                  colLabels=metrics_summary.columns,
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.25, 0.35, 0.35])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

ax9.set_title('Resumen de Métricas', fontsize=12, fontweight='bold')

plt.tight_layout()

# Guardar gráfico
plot_file = os.path.join(output_dir, 'comparativa_modelos_lightgbm_optimizado.png')
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Gráfico guardado en: {plot_file}")
plt.close()

# Crear Excel con resultados detallados
print("\nGenerando archivo Excel con resultados...")

excel_file = os.path.join(output_dir, 'resultados_comparativa_lasso.xlsx')
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Métricas comparativas
    metrics_comparison = pd.DataFrame({
        'Métrica': ['MAE Train', 'MAE Test', 'RMSE Train', 'RMSE Test', 
                   'R² Train', 'R² Test', 'MAPE Train (%)', 'MAPE Test (%)',
                   'MAE Picos Train', 'MAE Picos Test', 'Best Iteration'],
        'Modelo 2021-2025': [
            metrics_1['train_mae'], metrics_1['test_mae'],
            metrics_1['train_rmse'], metrics_1['test_rmse'],
            metrics_1['train_r2'], metrics_1['test_r2'],
            metrics_1['train_mape'], metrics_1['test_mape'],
            metrics_1['peak_mae_train'], metrics_1['peak_mae_test'],
            metrics_1['best_iteration']
        ],
        'Modelo 2022-2025': [
            metrics_2['train_mae'], metrics_2['test_mae'],
            metrics_2['train_rmse'], metrics_2['test_rmse'],
            metrics_2['train_r2'], metrics_2['test_r2'],
            metrics_2['train_mape'], metrics_2['test_mape'],
            metrics_2['peak_mae_train'], metrics_2['peak_mae_test'],
            metrics_2['best_iteration']
        ]
    })
    metrics_comparison.to_excel(writer, sheet_name='Métricas_Comparativas', index=False)
    
    # Predicciones modelo 1
    pred_df_1 = pd.DataFrame({
        'fecha': test_dates,
        'año': df[test_mask]['año'].values,
        'semana_epi': df[test_mask]['semana_epi'].values,
        'casos_reales': y_test,
        'predicciones_2021_2025': y_test_pred_1,
        'error_2021_2025': np.abs(y_test - y_test_pred_1)
    })
    pred_df_1.to_excel(writer, sheet_name='Predicciones_Modelo_1', index=False)
    
    # Predicciones modelo 2
    pred_df_2 = pd.DataFrame({
        'fecha': test_dates,
        'año': df[test_mask]['año'].values,
        'semana_epi': df[test_mask]['semana_epi'].values,
        'casos_reales': y_test,
        'predicciones_2022_2025': y_test_pred_2,
        'error_2022_2025': np.abs(y_test - y_test_pred_2)
    })
    pred_df_2.to_excel(writer, sheet_name='Predicciones_Modelo_2', index=False)
    
    # Importancia de features - Modelo 1
    feature_importance_df_1.to_excel(writer, sheet_name='Importancia_Features_M1', index=False)
    
    # Importancia de features - Modelo 2
    feature_importance_df_2.to_excel(writer, sheet_name='Importancia_Features_M2', index=False)
    
    # Parámetros del modelo
    params_df = pd.DataFrame({
        'Parámetro': list(model_params.keys()),
        'Valor': [str(v) for v in model_params.values()]
    })
    params_df.to_excel(writer, sheet_name='Parámetros_Optimizados', index=False)

print(f"Excel guardado en: {excel_file}")

# Guardar modelos
model_1.save_model(os.path.join(output_dir, 'modelo_2021_2025.txt'))
model_2.save_model(os.path.join(output_dir, 'modelo_2022_2025.txt'))

# Resumen final
print("\n" + "="*70)
print("RESUMEN FINAL - COMPARATIVA DE MODELOS")
print("="*70)
print(f"✓ Dataset reducido: {input_file}")
print(f"✓ Predictores: {len(predictor_cols)}")
print(f"✓ Optimización con Optuna: {len(best_params)} parámetros ajustados")
print("\nComparación de MAE en Test:")
print(f"  - Modelo 2021-2025: {metrics_1['test_mae']:.4f}")
print(f"  - Modelo 2022-2025: {metrics_2['test_mae']:.4f}")
print(f"  - Mejor modelo: {'2022-2025' if metrics_2['test_mae'] < metrics_1['test_mae'] else '2021-2025'}")
print(f"\nMAE en picos (Test):")
print(f"  - Modelo 2021-2025: {metrics_1['peak_mae_test']:.4f}")
print(f"  - Modelo 2022-2025: {metrics_2['peak_mae_test']:.4f}")
print(f"\nArchivos generados:")
print(f"  - Gráfico comparativo: {plot_file}")
print(f"  - Resultados Excel: {excel_file}")
print(f"  - Modelo 2021-2025: {os.path.join(output_dir, 'modelo_2021_2025.txt')}")
print(f"  - Modelo 2022-2025: {os.path.join(output_dir, 'modelo_2022_2025.txt')}")
print("="*70)

# Recomendación final
print("\nRECOMENDACIÓN:")
if metrics_2['test_mae'] < metrics_1['test_mae']:
    print("✓ El modelo entrenado con 2022-2025 es mejor (menor MAE)")
    print(f"✓ Mejora del {((metrics_1['test_mae'] - metrics_2['test_mae']) / metrics_1['test_mae'] * 100):.1f}% en MAE")
else:
    print("✓ El modelo entrenado con 2021-2025 es mejor (menor MAE)")
    print(f"✓ Mejora del {((metrics_2['test_mae'] - metrics_1['test_mae']) / metrics_2['test_mae'] * 100):.1f}% en MAE")
print("="*70)

[I 2026-07-30 19:51:06,715] A new study created in memory with name: no-name-259b8e75-742f-4a6a-96e3-98f347009fdb


Cargando datos reducidos...
Predictores: 19
Rango de fechas: 2021-03-28 00:00:00 a 2026-05-31 00:00:00

Preparando datos...
Periodo 1 - Entrenamiento 2021-2025: 249 registros
Periodo 2 - Entrenamiento 2022-2025: 209 registros
Test 2026: 21 registros

Optimizando hiperparámetros con validación temporal...


Best trial: 0. Best value: 14.5619:   0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[10]	train's l1: 3.7525	val's l1: 14.5619
[I 2026-07-30 19:51:06,761] Trial 0 finished with value: 14.56192190321612 and parameters: {'num_leaves': 44, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.8659969709057025, 'bagging_fraction': 0.7993292420985183, 'bagging_freq': 2, 'min_child_samples': 12, 'reg_alpha': 0.05808361216819946, 'reg_lambda': 0.8661761457749352, 'min_split_gain': 0.3005575058716044}. Best is trial 0 with value: 14.56192190321612.
Training until validation scores don't improve for 100 rounds


Best trial: 1. Best value: 13.133:   8%|▊         | 4/50 [00:00<00:04,  9.47it/s] 

...

Early stopping, best iteration is:
[181]	train's l1: 1.58351	val's l1: 12.4196
[I 2026-07-30 19:51:11,581] Trial 45 finished with value: 12.419554025904924 and parameters: {'num_leaves': 55, 'learning_rate': 0.020437946530117303, 'feature_fraction': 0.7843712230565377, 'bagging_fraction': 0.9130268859121407, 'bagging_freq': 10, 'min_child_samples': 7, 'reg_alpha': 0.3904374632854395, 'reg_lambda': 0.04170639414590947, 'min_split_gain': 0.2569846227012578}. Best is trial 43 with value: 11.870463445173177.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[88]	train's l1: 6.10662	val's l1: 19.3501
[I 2026-07-30 19:51:11,652] Trial 46 finished with value: 19.3501388721089 and parameters: {'num_leaves': 59, 'learning_rate': 0.033503159930457264, 'feature_fraction': 0.7352950560519714, 'bagging_fraction': 0.8076988066388051, 'bagging_freq': 10, 'min_child_samples': 44, 'reg_alpha': 0.4288304040884504, 'reg_lambda': 0.00412466963634292, 'min_spl

Best trial: 43. Best value: 11.8705:  98%|█████████▊| 49/50 [00:05<00:00,  7.29it/s]

Early stopping, best iteration is:
[180]	train's l1: 2.75043	val's l1: 13.4999
[I 2026-07-30 19:51:11,812] Trial 47 finished with value: 13.499916656628606 and parameters: {'num_leaves': 72, 'learning_rate': 0.016409094820017237, 'feature_fraction': 0.740214438372807, 'bagging_fraction': 0.9151806414801927, 'bagging_freq': 10, 'min_child_samples': 11, 'reg_alpha': 0.3326845901074443, 'reg_lambda': 0.11778160966252788, 'min_split_gain': 0.3956117895366544}. Best is trial 43 with value: 11.870463445173177.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[147]	train's l1: 3.79125	val's l1: 13.3145
[I 2026-07-30 19:51:11,925] Trial 48 finished with value: 13.314510687587337 and parameters: {'num_leaves': 46, 'learning_rate': 0.020794640498808886, 'feature_fraction': 0.8044765857720902, 'bagging_fraction': 0.8655784333729255, 'bagging_freq': 9, 'min_child_samples': 18, 'reg_alpha': 0.23669477957151708, 'reg_lambda': 0.038703836947117314, 'min

Best trial: 43. Best value: 11.8705: 100%|██████████| 50/50 [00:05<00:00,  9.26it/s]


Early stopping, best iteration is:
[161]	train's l1: 1.50125	val's l1: 12.7076
[I 2026-07-30 19:51:12,118] Trial 49 finished with value: 12.707647968127835 and parameters: {'num_leaves': 40, 'learning_rate': 0.023796416683210773, 'feature_fraction': 0.897002232506768, 'bagging_fraction': 0.9312478361807239, 'bagging_freq': 9, 'min_child_samples': 7, 'reg_alpha': 0.3962680553238223, 'reg_lambda': 0.23297865716116542, 'min_split_gain': 0.47642328478655127}. Best is trial 43 with value: 11.870463445173177.

Mejores parámetros encontrados:
  num_leaves: 58
  learning_rate: 0.03507645979363265
  feature_fraction: 0.9380908681298546
  bagging_fraction: 0.9164717113554568
  bagging_freq: 10
  min_child_samples: 8
  reg_alpha: 0.4078688702657929
  reg_lambda: 0.06632658566453858
  min_split_gain: 0.4286341165136861
Mejor MAE en validación: 11.8705

Entrenando modelos con los mejores parámetros...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[

In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from scipy import stats
from sklearn.preprocessing import QuantileTransformer
import optuna

# Configuración
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados\datos_reducidos_lasso.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
os.makedirs(output_dir, exist_ok=True)

# Cargar datos
print("Cargando datos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])

# Identificar columnas
target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']
predictor_cols = [col for col in df.columns if col not in exclude_cols + [target_col]]

print(f"Predictores: {len(predictor_cols)}")

# Función para crear pesos enfocados en picos
def create_peak_weights(y, peak_threshold=0.85, extreme_weight=3.0, high_weight=1.5):
    """
    Crea pesos para dar más importancia a los picos
    """
    weights = np.ones_like(y, dtype=float)
    
    # Identificar picos usando percentiles
    threshold_high = np.percentile(y, peak_threshold * 100)
    threshold_extreme = np.percentile(y, 95)
    
    # Asignar pesos
    extreme_mask = y >= threshold_extreme
    high_mask = (y >= threshold_high) & (~extreme_mask)
    
    weights[extreme_mask] = extreme_weight
    weights[high_mask] = high_weight
    
    # También dar peso adicional a semanas con aumento significativo
    if len(y) > 1:
        diff = np.diff(y)
        diff_high = np.percentile(np.abs(diff), 85)
        increase_mask = np.concatenate([np.array([False]), (diff > diff_high)])
        weights[increase_mask] *= 1.3
    
    return weights

# Función de pérdida personalizada para enfocarse en picos
def peak_focused_loss(y_true, y_pred, weight):
    """
    Pérdida que da más peso a los errores en picos
    """
    errors = y_true - y_pred
    squared_errors = errors ** 2
    
    # Pérdida estándar MSE con pesos
    weighted_mse = np.mean(weight * squared_errors)
    
    # Penalización adicional por subestimar picos
    peak_mask = y_true > np.percentile(y_true, 80)
    under_penalty = np.mean(np.maximum(0, y_true[peak_mask] - y_pred[peak_mask])) * 0.5
    
    return weighted_mse + under_penalty

# Función para entrenar modelo con enfoque en picos
def train_peak_focused_model(X_train, y_train, X_test, y_test, 
                           train_dates, test_dates, model_name, 
                           peak_weight=3.0, custom_params=None):
    """
    Entrena modelo enfocado en predicción de picos
    """
    
    # Crear pesos para entrenamiento
    train_weights = create_peak_weights(y_train, peak_threshold=0.85, 
                                       extreme_weight=peak_weight, high_weight=1.5)
    
    # Parámetros base
    if custom_params is None:
        params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'num_leaves': 45,
            'learning_rate': 0.03,
            'feature_fraction': 0.85,
            'bagging_fraction': 0.85,
            'bagging_freq': 5,
            'min_child_samples': 15,
            'reg_alpha': 0.3,
            'reg_lambda': 0.3,
            'min_split_gain': 0.1,
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42,
            'max_depth': 8  # Limitar profundidad para evitar overfitting
        }
    else:
        params = custom_params
    
    # Crear datasets con pesos
    dtrain = lgb.Dataset(X_train, y_train, weight=train_weights)
    dtest = lgb.Dataset(X_test, y_test, reference=dtrain)
    
    # Entrenar
    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dtrain, dtest],
        valid_names=['train', 'test'],
        num_boost_round=3000,
        callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)]
    )
    
    # Predicciones
    y_train_pred = model.predict(X_train, num_iteration=model.best_iteration)
    y_test_pred = model.predict(X_test, num_iteration=model.best_iteration)
    
    return model, y_train_pred, y_test_pred, train_weights

# Función para optimizar parámetros con enfoque en picos
def optimize_for_peaks(X_train, y_train, X_val, y_val):
    """
    Optimiza parámetros para minimizar error en picos
    """
    
    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'num_leaves': trial.suggest_int('num_leaves', 20, 80),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
            'max_depth': trial.suggest_int('max_depth', 5, 12),
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42
        }
        
        # Crear pesos enfocados en picos para validación
        val_weights = create_peak_weights(y_val, peak_threshold=0.85, 
                                        extreme_weight=3.0, high_weight=1.5)
        
        dtrain = lgb.Dataset(X_train, y_train)
        dval = lgb.Dataset(X_val, y_val, weight=val_weights, reference=dtrain)
        
        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dtrain, dval],
            valid_names=['train', 'val'],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        
        y_pred = model.predict(X_val, num_iteration=model.best_iteration)
        
        # Métrica personalizada: MAE en picos
        peak_mask = y_val > np.percentile(y_val, 80)
        if np.sum(peak_mask) > 0:
            peak_mae = mean_absolute_error(y_val[peak_mask], y_pred[peak_mask])
        else:
            peak_mae = mean_absolute_error(y_val, y_pred)
        
        # Penalizar subestimación de picos
        under_penalty = np.mean(np.maximum(0, y_val[peak_mask] - y_pred[peak_mask])) if np.sum(peak_mask) > 0 else 0
        
        return peak_mae + under_penalty * 0.5
    
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    print(f"\nMejores parámetros para picos:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study.best_params

# Preparar datos
print("\nPreparando datos...")
train_mask = df['año'].isin([2021, 2022, 2023, 2024, 2025])
test_mask = df['año'] == 2026

X_train = df[train_mask][predictor_cols].values
y_train = df[train_mask][target_col].values
X_test = df[test_mask][predictor_cols].values
y_test = df[test_mask][target_col].values
train_dates = df[train_mask]['fecha'].values
test_dates = df[test_mask]['fecha'].values

# Optimizar parámetros con validación 2024-2025
print("\nOptimizando parámetros para mejora de picos...")
opt_train_mask = df['año'].isin([2021, 2022, 2023, 2024])
opt_val_mask = df['año'] == 2025

X_opt_train = df[opt_train_mask][predictor_cols].values
y_opt_train = df[opt_train_mask][target_col].values
X_opt_val = df[opt_val_mask][predictor_cols].values
y_opt_val = df[opt_val_mask][target_col].values

best_params = optimize_for_peaks(X_opt_train, y_opt_train, X_opt_val, y_opt_val)

# Probar diferentes pesos para picos
print("\nProbando diferentes pesos para picos...")
peak_weights = [2.0, 3.0, 4.0, 5.0, 7.0]
results = {}

for weight in peak_weights:
    print(f"\nEntrenando con peso para picos = {weight}")
    model, y_train_pred, y_test_pred, train_weights = train_peak_focused_model(
        X_train, y_train, X_test, y_test,
        train_dates, test_dates,
        f"peak_weight_{weight}",
        peak_weight=weight,
        custom_params=best_params
    )
    
    # Métricas
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Métricas en picos (top 80%)
    peak_mask_test = y_test > np.percentile(y_test, 80)
    peak_mae_test = mean_absolute_error(y_test[peak_mask_test], y_test_pred[peak_mask_test]) if np.sum(peak_mask_test) > 0 else np.nan
    
    results[weight] = {
        'model': model,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'test_r2': test_r2,
        'peak_mae_test': peak_mae_test
    }
    
    print(f"  Test MAE: {test_mae:.2f}")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Peak MAE: {peak_mae_test:.2f}")

# Seleccionar mejor modelo (minimizar peak MAE)
best_weight = min(results.keys(), key=lambda k: results[k]['peak_mae_test'])
best_model = results[best_weight]['model']
y_test_pred_best = results[best_weight]['y_test_pred']
y_train_pred_best = results[best_weight]['y_train_pred']

print(f"\nMejor peso para picos: {best_weight}")

# Crear gráficos mejorados
print("\nGenerando gráficos...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Modelo LightGBM Enfocado en Picos - Reducción Lasso', fontsize=16, fontweight='bold')

# 1. Series temporales - Entrenamiento
ax1 = axes[0, 0]
ax1.plot(train_dates, y_train, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax1.plot(train_dates, y_train_pred_best, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
# Resaltar picos en entrenamiento
peak_mask_train = y_train > np.percentile(y_train, 80)
ax1.scatter(train_dates[peak_mask_train], y_train[peak_mask_train], 
           color='gold', s=50, label='Picos (>80%)', zorder=5, alpha=0.8)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos Dengue')
ax1.set_title(f'Entrenamiento 2021-2025\nMAE: {results[best_weight]["train_mae"]:.2f}')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. Series temporales - Test
ax2 = axes[0, 1]
ax2.plot(test_dates, y_test, 'b-', label='Real', linewidth=2, marker='o', markersize=6)
ax2.plot(test_dates, y_test_pred_best, 'r-', label=f'Predicho (MAE: {results[best_weight]["test_mae"]:.2f})', 
         linewidth=2, marker='s', markersize=5, alpha=0.8)
# Resaltar picos en test
peak_mask_test = y_test > np.percentile(y_test, 80)
ax2.scatter(test_dates[peak_mask_test], y_test[peak_mask_test], 
           color='gold', s=100, label='Picos (>80%)', zorder=5, alpha=0.8)
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos Dengue')
ax2.set_title(f'Test 2026 - Enfoque en Picos\nPeak MAE: {results[best_weight]["peak_mae_test"]:.2f}')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Comparación con modelo anterior
ax3 = axes[0, 2]
# Datos simulados del modelo anterior para comparación (usando resultados previos)
ax3.plot(test_dates, y_test, 'k-', label='Real', linewidth=2)
ax3.plot(test_dates, y_test_pred_best, 'r-', label='Nuevo Modelo (Picos)', linewidth=2)
# Usar líneas para mostrar mejora
ax3.set_xlabel('Fecha')
ax3.set_ylabel('Casos Dengue')
ax3.set_title(f'Mejora en Picos\nPeak MAE reducido a {results[best_weight]["peak_mae_test"]:.2f}')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# 4. Error en picos vs general
ax4 = axes[1, 0]
# Comparar errores
categories = ['General', 'Picos (top 80%)']
errors_general = mean_absolute_error(y_test, y_test_pred_best)
errors_peak = results[best_weight]['peak_mae_test']

bars = ax4.bar(categories, [errors_general, errors_peak], color=['skyblue', 'coral'])
ax4.axhline(y=np.mean([errors_general, errors_peak]), color='red', linestyle='--', alpha=0.7)
ax4.set_ylabel('MAE')
ax4.set_title('Comparación de Errores')
ax4.grid(True, alpha=0.3)

# Añadir valores en las barras
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{height:.2f}', ha='center', va='bottom')

# 5. Scatter plot con énfasis en picos
ax5 = axes[1, 1]
scatter = ax5.scatter(y_test, y_test_pred_best, c=y_test, cmap='viridis', alpha=0.6, s=50)
ax5.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
# Resaltar picos
peak_mask = y_test > np.percentile(y_test, 80)
ax5.scatter(y_test[peak_mask], y_test_pred_best[peak_mask], 
           color='red', s=100, label='Picos', edgecolor='black', zorder=5)
ax5.set_xlabel('Valores Reales')
ax5.set_ylabel('Predicciones')
ax5.set_title(f'Scatter Plot - Test\nR²: {results[best_weight]["test_r2"]:.4f}')
ax5.legend()
ax5.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax5, label='Casos Reales')

# 6. Importancia de características
ax6 = axes[1, 2]
importance = best_model.feature_importance(importance_type='gain')
feature_importance_df = pd.DataFrame({
    'Feature': predictor_cols,
    'Importance': importance
}).sort_values('Importance', ascending=True)

top_features = feature_importance_df.tail(10)
ax6.barh(top_features['Feature'], top_features['Importance'], color='teal', alpha=0.7)
ax6.set_xlabel('Importancia (Gain)')
ax6.set_title('Top 10 Características')
ax6.grid(True, alpha=0.3)

plt.tight_layout()

# Guardar gráfico
plot_file = os.path.join(output_dir, 'modelo_enfocado_picos.png')
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Gráfico guardado en: {plot_file}")
plt.close()

# Generar archivo Excel con resultados detallados
print("\nGenerando Excel con resultados...")
excel_file = os.path.join(output_dir, 'resultados_enfocado_picos.xlsx')

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Resumen de todos los pesos probados
    summary_df = pd.DataFrame({
        'Peso_Picos': list(results.keys()),
        'Train_MAE': [results[w]['train_mae'] for w in results],
        'Test_MAE': [results[w]['test_mae'] for w in results],
        'Test_R2': [results[w]['test_r2'] for w in results],
        'Peak_MAE': [results[w]['peak_mae_test'] for w in results]
    })
    summary_df.to_excel(writer, sheet_name='Resumen_Pesos', index=False)
    
    # Predicciones detalladas para el mejor modelo
    pred_df = pd.DataFrame({
        'fecha': test_dates,
        'año': df[test_mask]['año'].values,
        'semana_epi': df[test_mask]['semana_epi'].values,
        'casos_reales': y_test,
        'predicciones': y_test_pred_best,
        'error': np.abs(y_test - y_test_pred_best),
        'es_pico': y_test > np.percentile(y_test, 80)
    })
    pred_df.to_excel(writer, sheet_name='Predicciones_Mejor', index=False)
    
    # Características importantes
    feature_importance_df.to_excel(writer, sheet_name='Importancia_Features', index=False)
    
    # Parámetros del modelo
    params_df = pd.DataFrame({
        'Parámetro': list(best_params.keys()),
        'Valor': [str(v) for v in best_params.values()]
    })
    params_df.to_excel(writer, sheet_name='Parámetros_Optimizados', index=False)
    
    # Análisis de errores por rango de casos
    error_analysis = pd.DataFrame()
    bins = [0, 5, 10, 20, 50, 100, 200]
    labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100-200']
    pred_df['rango_casos'] = pd.cut(pred_df['casos_reales'], bins=bins, labels=labels)
    
    for label in labels:
        mask = pred_df['rango_casos'] == label
        if mask.sum() > 0:
            error_analysis.loc[label, 'Count'] = mask.sum()
            error_analysis.loc[label, 'MAE'] = pred_df[mask]['error'].mean()
            error_analysis.loc[label, 'Max_Error'] = pred_df[mask]['error'].max()
    
    error_analysis.to_excel(writer, sheet_name='Análisis_Errores')

print(f"Excel guardado en: {excel_file}")

# Guardar modelo
model_file = os.path.join(output_dir, 'modelo_enfocado_picos.txt')
best_model.save_model(model_file)
print(f"Modelo guardado en: {model_file}")

# Resumen final
print("\n" + "="*70)
print("RESUMEN - MODELO ENFOCADO EN PICOS")
print("="*70)
print(f"✓ Mejor peso para picos: {best_weight}")
print(f"✓ MAE General: {results[best_weight]['test_mae']:.2f}")
print(f"✓ MAE en Picos: {results[best_weight]['peak_mae_test']:.2f}")
print(f"✓ R² en Test: {results[best_weight]['test_r2']:.4f}")

# Calcular mejora relativa
improvement = ((12.65 - results[best_weight]['peak_mae_test']) / 12.65) * 100
print(f"\n✓ Mejora en MAE de picos: {improvement:.1f}%")
print(f"  (Anterior: 12.65 → Nuevo: {results[best_weight]['peak_mae_test']:.2f})")

print("\nEstrategias implementadas:")
print("1. Pesos personalizados para dar más importancia a picos")
print("2. Optimización específica de parámetros para picos")
print("3. Pérdida personalizada penalizando subestimación")
print("4. Análisis de errores por rango de casos")
print("5. Múltiples pesos probados (2.0, 3.0, 4.0, 5.0, 7.0)")
print("="*70)

# Recomendaciones
print("\nRECOMENDACIONES ADICIONALES:")
print("1. Considerar usar ensemble de modelos con diferentes pesos")
print("2. Evaluar si se necesita incluir más variables rezagadas")
print("3. Probar transformación logarítmica para casos extremos")
print("4. Considerar modelo específico para semanas epidémicas")
print("5. Revisar si hay variables adicionales que expliquen picos")
print("="*70)

[I 2026-07-30 20:01:13,344] A new study created in memory with name: no-name-54cdd20d-9743-433f-ad48-a2c849d57795


Cargando datos...
Predictores: 19

Preparando datos...

Optimizando parámetros para mejora de picos...


Best trial: 0. Best value: 42.3805:   2%|▏         | 1/50 [00:00<00:05,  8.94it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[63]	train's l1: 1.69872	val's l1: 16.6728
[I 2026-07-30 20:01:13,454] Trial 0 finished with value: 42.38049219224528 and parameters: {'num_leaves': 42, 'learning_rate': 0.08927180304353628, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8394633936788146, 'bagging_freq': 2, 'min_child_samples': 10, 'reg_alpha': 0.05808361216819946, 'reg_lambda': 0.8661761457749352, 'min_split_gain': 0.3005575058716044, 'max_depth': 10}. Best is trial 0 with value: 42.38049219224528.
Training until validation scores don't improve for 100 rounds


Best trial: 0. Best value: 42.3805:   2%|▏         | 1/50 [00:00<00:05,  8.94it/s]

...

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[81]	train's l1: 1.09999	val's l1: 14.9017
[I 2026-07-30 20:01:20,828] Trial 46 finished with value: 36.37203835103835 and parameters: {'num_leaves': 45, 'learning_rate': 0.08819398240449272, 'feature_fraction': 0.8357255140549688, 'bagging_fraction': 0.6185843997440421, 'bagging_freq': 6, 'min_child_samples': 5, 'reg_alpha': 0.1543503528720706, 'reg_lambda': 0.89928343542261, 'min_split_gain': 0.1249555211712281, 'max_depth': 10}. Best is trial 24 with value: 34.577723177457635.
Training until validation scores don't improve for 100 rounds


Best trial: 24. Best value: 34.5777:  96%|█████████▌| 48/50 [00:07<00:00,  6.52it/s]

Early stopping, best iteration is:
[63]	train's l1: 5.00191	val's l1: 21.9264
[I 2026-07-30 20:01:20,927] Trial 47 finished with value: 57.7925141196491 and parameters: {'num_leaves': 47, 'learning_rate': 0.0898755857887271, 'feature_fraction': 0.9527334724131439, 'bagging_fraction': 0.6339221009645246, 'bagging_freq': 7, 'min_child_samples': 27, 'reg_alpha': 0.14899953600748925, 'reg_lambda': 0.7944135990716553, 'min_split_gain': 0.12673862221785623, 'max_depth': 8}. Best is trial 24 with value: 34.577723177457635.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[77]	train's l1: 1.08279	val's l1: 15.3715


Best trial: 24. Best value: 34.5777: 100%|██████████| 50/50 [00:07<00:00,  6.34it/s]


[I 2026-07-30 20:01:21,123] Trial 48 finished with value: 38.924601759654514 and parameters: {'num_leaves': 79, 'learning_rate': 0.07447553403578781, 'feature_fraction': 0.8623333308993384, 'bagging_fraction': 0.8971207337636383, 'bagging_freq': 7, 'min_child_samples': 7, 'reg_alpha': 0.6385134481083221, 'reg_lambda': 0.8378306628321159, 'min_split_gain': 0.38956317256258244, 'max_depth': 11}. Best is trial 24 with value: 34.577723177457635.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[51]	train's l1: 3.78326	val's l1: 16.6938
[I 2026-07-30 20:01:21,225] Trial 49 finished with value: 43.99609561362553 and parameters: {'num_leaves': 51, 'learning_rate': 0.09049533645885266, 'feature_fraction': 0.8170771755666558, 'bagging_fraction': 0.6621889742351776, 'bagging_freq': 10, 'min_child_samples': 14, 'reg_alpha': 0.10135293491879549, 'reg_lambda': 0.9537965441823183, 'min_split_gain': 0.08666493920441885, 'max_depth': 12}. Best is trial 2

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')  # Usar backend sin interfaz gráfica
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PowerTransformer
import gc

# Configuración
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados\datos_reducidos_lasso.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
os.makedirs(output_dir, exist_ok=True)

# Configurar para liberar memoria
os.environ['OMP_NUM_THREADS'] = '4'

print("Cargando datos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])

# Identificar columnas
target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']
predictor_cols = [col for col in df.columns if col not in exclude_cols + [target_col]]

print(f"Predictores originales: {len(predictor_cols)}")

# ==================== 1. LIMPIEZA DE DATOS ====================
print("\n1. Limpiando datos...")

def clean_data(df):
    """Limpia valores infinitos y extremos"""
    df_clean = df.copy()
    for col in df_clean.columns:
        if df_clean[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
            median_val = df_clean[col].median()
            if pd.isna(median_val):
                median_val = 0
            df_clean[col] = df_clean[col].fillna(median_val)
            upper_limit = df_clean[col].quantile(0.999)
            lower_limit = df_clean[col].quantile(0.001)
            if not pd.isna(upper_limit) and not pd.isna(lower_limit):
                df_clean[col] = df_clean[col].clip(lower=lower_limit, upper=upper_limit)
    return df_clean

df = clean_data(df)

# ==================== 2. FEATURE ENGINEERING OPTIMIZADO ====================
print("\n2. Creando nuevas características (optimizado)...")

def create_advanced_features_optimized(df, target_col):
    """Crea características avanzadas optimizadas para memoria"""
    df_enhanced = df.copy()
    
    # Lag 1-12 (solo algunos)
    for lag in [1, 2, 3, 4, 6, 8, 12]:
        if lag <= 12:
            df_enhanced[f'{target_col}_lag_{lag}'] = df_enhanced[target_col].shift(lag)
    
    # Medias móviles
    for window in [3, 4, 6, 8]:
        df_enhanced[f'{target_col}_mean_{window}'] = df_enhanced[target_col].rolling(window=window, min_periods=1).mean()
    
    # Diferencias
    for lag in [1, 2, 3, 4]:
        df_enhanced[f'{target_col}_diff_{lag}'] = df_enhanced[target_col].diff(lag)
    
    # Z-scores
    for window in [4, 8]:
        rolling_mean = df_enhanced[target_col].rolling(window=window, min_periods=1).mean()
        rolling_std = df_enhanced[target_col].rolling(window=window, min_periods=1).std()
        df_enhanced[f'{target_col}_zscore_{window}'] = (df_enhanced[target_col] - rolling_mean) / (rolling_std + 0.1)
        df_enhanced[f'{target_col}_zscore_{window}'] = df_enhanced[f'{target_col}_zscore_{window}'].replace([np.inf, -np.inf], 0)
        df_enhanced[f'{target_col}_zscore_{window}'] = df_enhanced[f'{target_col}_zscore_{window}'].fillna(0)
    
    # Características estacionales
    if 'semana_epi' in df_enhanced.columns:
        df_enhanced['sin_semana'] = np.sin(2 * np.pi * df_enhanced['semana_epi'] / 52)
        df_enhanced['cos_semana'] = np.cos(2 * np.pi * df_enhanced['semana_epi'] / 52)
    
    return df_enhanced

df_enhanced = create_advanced_features_optimized(df, target_col)
df_enhanced = clean_data(df_enhanced)

# Eliminar filas con NaN
df_enhanced = df_enhanced.dropna()
print(f"Dimensiones después de feature engineering: {df_enhanced.shape}")

# Identificar nuevas columnas predictoras
new_predictor_cols = [col for col in df_enhanced.columns if col not in exclude_cols + [target_col]]
print(f"Nuevos predictores: {len(new_predictor_cols)}")

# ==================== 3. TRANSFORMACIÓN DE TARGET ====================
print("\n3. Aplicando transformación al target...")

target_positive = df_enhanced[target_col] + 1
target_positive = target_positive.clip(lower=0.001)

pt = PowerTransformer(method='box-cox', standardize=False)
y_transformed = pt.fit_transform(target_positive.values.reshape(-1, 1))
df_enhanced['target_transformed'] = y_transformed.flatten()
df_enhanced['target_transformed'] = df_enhanced['target_transformed'].replace([np.inf, -np.inf], 0)
df_enhanced['target_transformed'] = df_enhanced['target_transformed'].fillna(0)

# ==================== 4. SELECCIÓN DE PREDICTORES ====================
print("\n4. Seleccionando predictores más relevantes...")

X_temp = df_enhanced[new_predictor_cols].values
y_temp = df_enhanced['target_transformed'].values

X_temp = np.nan_to_num(X_temp, nan=0.0, posinf=0.0, neginf=0.0)
y_temp = np.nan_to_num(y_temp, nan=0.0, posinf=0.0, neginf=0.0)

# Usar Random Forest para seleccionar top predictores
print("  Seleccionando con Random Forest...")
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=8)
rf.fit(X_temp, y_temp)

importance_df = pd.DataFrame({
    'Feature': new_predictor_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Seleccionar top 30 predictores
top_n = min(30, len(new_predictor_cols))
selected_features = importance_df.head(top_n)['Feature'].tolist()
print(f"Top {top_n} predictores seleccionados de {len(new_predictor_cols)}")

# Liberar memoria
del X_temp, y_temp, rf, importance_df
gc.collect()

# ==================== 5. PREPARACIÓN DE DATOS ====================
print("\n5. Preparando datos para entrenamiento...")

X = df_enhanced[selected_features].values
y = df_enhanced['target_transformed'].values
y_original = df_enhanced[target_col].values

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
y_original = np.nan_to_num(y_original, nan=0.0, posinf=0.0, neginf=0.0)

# Separar por años
train_mask = df_enhanced['año'].isin([2021, 2022, 2023, 2024, 2025])
test_mask = df_enhanced['año'] == 2026

X_train = X[train_mask]
X_test = X[test_mask]
y_train_trans = y[train_mask]
y_test_trans = y[test_mask]
y_train_orig = y_original[train_mask]
y_test_orig = y_original[test_mask]

train_dates = df_enhanced[train_mask]['fecha'].values
test_dates = df_enhanced[test_mask]['fecha'].values

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# ==================== 6. FUNCIONES DE PESO ====================
def create_advanced_weights(y_orig, y_trans, peak_threshold=0.80):
    """Sistema de pesos para enfocarse en picos"""
    weights = np.ones(len(y_orig))
    
    quantiles = [0.7, 0.8, 0.9, 0.95, 0.99]
    weight_multipliers = [1.5, 2.0, 3.0, 5.0, 8.0]
    
    for q, mult in zip(quantiles, weight_multipliers):
        threshold = np.percentile(y_orig, q * 100)
        mask = y_orig >= threshold
        weights[mask] *= mult
    
    weights = weights / np.mean(weights)
    weights = np.clip(weights, 0.1, 10.0)
    return weights

# ==================== 7. MODELO ENSEMBLE SIMPLIFICADO ====================
def train_simple_ensemble(X_train, y_train_trans, y_train_orig, 
                         X_test, y_test_trans, y_test_orig):
    """Entrena un ensemble simple para mayor velocidad"""
    
    print("\n  Entrenando modelo 1: LightGBM con pesos")
    weights_agg = create_advanced_weights(y_train_orig, y_train_trans, peak_threshold=0.75)
    
    params_agg = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.03,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 15,
        'reg_alpha': 0.3,
        'reg_lambda': 0.3,
        'max_depth': 7,
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42
    }
    
    dtrain1 = lgb.Dataset(X_train, y_train_trans, weight=weights_agg)
    dtest1 = lgb.Dataset(X_test, y_test_trans, reference=dtrain1)
    
    model1 = lgb.train(
        params_agg,
        dtrain1,
        valid_sets=[dtrain1, dtest1],
        valid_names=['train', 'test'],
        num_boost_round=1500,
        callbacks=[lgb.early_stopping(100)]
    )
    
    pred_test1 = model1.predict(X_test, num_iteration=model1.best_iteration)
    pred_train1 = model1.predict(X_train, num_iteration=model1.best_iteration)
    
    print("  Entrenando modelo 2: LightGBM estándar")
    params_std = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 123
    }
    
    dtrain2 = lgb.Dataset(X_train, y_train_trans)
    dtest2 = lgb.Dataset(X_test, y_test_trans, reference=dtrain2)
    
    model2 = lgb.train(
        params_std,
        dtrain2,
        valid_sets=[dtrain2, dtest2],
        valid_names=['train', 'test'],
        num_boost_round=1500,
        callbacks=[lgb.early_stopping(100)]
    )
    
    pred_test2 = model2.predict(X_test, num_iteration=model2.best_iteration)
    pred_train2 = model2.predict(X_train, num_iteration=model2.best_iteration)
    
    print("  Creando predicción ensemble (promedio ponderado)...")
    final_train_pred = (pred_train1 * 0.6 + pred_train2 * 0.4)
    final_test_pred = (pred_test1 * 0.6 + pred_test2 * 0.4)
    
    # Invertir transformación
    final_train_pred_orig = pt.inverse_transform(final_train_pred.reshape(-1, 1)).flatten() - 1
    final_test_pred_orig = pt.inverse_transform(final_test_pred.reshape(-1, 1)).flatten() - 1
    
    final_train_pred_orig = np.clip(final_train_pred_orig, 0, None)
    final_test_pred_orig = np.clip(final_test_pred_orig, 0, None)
    
    return {
        'models': [model1, model2],
        'final_train_pred': final_train_pred_orig,
        'final_test_pred': final_test_pred_orig,
        'y_train_orig': y_train_orig,
        'y_test_orig': y_test_orig
    }

# ==================== 8. ENTRENAMIENTO ====================
print("\n" + "="*70)
print("ENTRENANDO MODELO ENSEMBLE...")
print("="*70)

try:
    results = train_simple_ensemble(
        X_train, y_train_trans, y_train_orig,
        X_test, y_test_trans, y_test_orig
    )
except Exception as e:
    print(f"Error en ensemble, usando modelo simple: {e}")
    # Modelo simple como fallback
    params_simple = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42
    }
    dtrain = lgb.Dataset(X_train, y_train_trans)
    dtest = lgb.Dataset(X_test, y_test_trans, reference=dtrain)
    model = lgb.train(params_simple, dtrain, valid_sets=[dtrain, dtest],
                     num_boost_round=1000, callbacks=[lgb.early_stopping(50)])
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred_orig = pt.inverse_transform(y_pred.reshape(-1, 1)).flatten() - 1
    results = {
        'models': [model],
        'final_train_pred': np.clip(pt.inverse_transform(model.predict(X_train).reshape(-1, 1)).flatten() - 1, 0, None),
        'final_test_pred': np.clip(y_pred_orig, 0, None),
        'y_train_orig': y_train_orig,
        'y_test_orig': y_test_orig
    }

# ==================== 9. MÉTRICAS ====================
def calculate_metrics(y_true, y_pred):
    y_true = np.nan_to_num(y_true, nan=0.0)
    y_pred = np.nan_to_num(y_pred, nan=0.0)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Picos
    peak_mask = y_true > np.percentile(y_true, 80)
    peak_mae = mean_absolute_error(y_true[peak_mask], y_pred[peak_mask]) if np.sum(peak_mask) > 0 else np.nan
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'peak_mae': peak_mae}

train_metrics = calculate_metrics(results['y_train_orig'], results['final_train_pred'])
test_metrics = calculate_metrics(results['y_test_orig'], results['final_test_pred'])

print("\n" + "="*70)
print("RESULTADOS DEL MODELO")
print("="*70)
print(f"Train MAE: {train_metrics['mae']:.2f} | R²: {train_metrics['r2']:.4f}")
print(f"Test MAE: {test_metrics['mae']:.2f} | R²: {test_metrics['r2']:.4f}")
print(f"Test Peak MAE (80%): {test_metrics['peak_mae']:.2f}")
print("="*70)

# ==================== 10. VISUALIZACIONES ====================
print("\n10. Generando visualizaciones...")

# Configurar estilo
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Modelo LightGBM Optimizado - Predicción Dengue', fontsize=14, fontweight='bold')

# 1. Series temporales - Test
ax1 = axes[0, 0]
ax1.plot(test_dates, results['y_test_orig'], 'b-', label='Real', linewidth=1.5, alpha=0.8)
ax1.plot(test_dates, results['final_test_pred'], 'r-', label='Predicho', linewidth=1.5, alpha=0.8)
peak_mask = results['y_test_orig'] > np.percentile(results['y_test_orig'], 80)
ax1.scatter(test_dates[peak_mask], results['y_test_orig'][peak_mask], 
           color='gold', s=40, label='Picos', zorder=5, alpha=0.7)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos')
ax1.set_title(f'Test 2026\nMAE: {test_metrics["mae"]:.2f} | Peak MAE: {test_metrics["peak_mae"]:.2f}')
ax1.legend(loc='upper left', fontsize=8)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# 2. Scatter plot
ax2 = axes[0, 1]
ax2.scatter(results['y_test_orig'], results['final_test_pred'], alpha=0.5, s=30, c='steelblue')
ax2.plot([0, max(results['y_test_orig'])], [0, max(results['y_test_orig'])], 'k--', alpha=0.5)
ax2.scatter(results['y_test_orig'][peak_mask], results['final_test_pred'][peak_mask], 
           color='red', s=80, label='Picos', edgecolor='black', zorder=5)
ax2.set_xlabel('Reales')
ax2.set_ylabel('Predichos')
ax2.set_title(f'R²: {test_metrics["r2"]:.4f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Importancia de características (top 10)
ax3 = axes[1, 0]
importance = results['models'][0].feature_importance(importance_type='gain')
fi_df = pd.DataFrame({'Feature': selected_features, 'Importance': importance})
fi_df = fi_df.sort_values('Importance', ascending=True).tail(10)
ax3.barh(fi_df['Feature'], fi_df['Importance'], color='darkblue', alpha=0.7)
ax3.set_xlabel('Importancia')
ax3.set_title('Top 10 Características')
ax3.grid(True, alpha=0.3)

# 4. Métricas comparativas
ax4 = axes[1, 1]
metrics_names = ['MAE', 'RMSE', 'R²']
train_vals = [train_metrics['mae'], train_metrics['rmse'], train_metrics['r2']]
test_vals = [test_metrics['mae'], test_metrics['rmse'], test_metrics['r2']]
x = np.arange(len(metrics_names))
width = 0.35
ax4.bar(x - width/2, train_vals, width, label='Train', color='skyblue')
ax4.bar(x + width/2, test_vals, width, label='Test', color='coral')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics_names)
ax4.set_title('Comparación Métricas')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()

# Guardar gráfico (CORREGIDO - sin optimize=True)
plot_file = os.path.join(output_dir, 'modelo_optimizado_resultados.png')
plt.savefig(plot_file, dpi=200, bbox_inches='tight')  # <- Remove optimize=True
print(f"Gráfico guardado en: {plot_file}")
plt.close()

# Liberar memoria
gc.collect()

# ==================== 11. GUARDAR RESULTADOS ====================
print("\n11. Guardando resultados...")

excel_file = os.path.join(output_dir, 'resultados_optimizados.xlsx')

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Métricas
    metrics_df = pd.DataFrame({
        'Métrica': ['MAE', 'RMSE', 'R²', 'Peak_MAE_80'],
        'Train': [train_metrics['mae'], train_metrics['rmse'], train_metrics['r2'], np.nan],
        'Test': [test_metrics['mae'], test_metrics['rmse'], test_metrics['r2'], test_metrics['peak_mae']]
    })
    metrics_df.to_excel(writer, sheet_name='Métricas', index=False)
    
    # Predicciones
    pred_df = pd.DataFrame({
        'fecha': test_dates,
        'año': df_enhanced[test_mask]['año'].values,
        'semana_epi': df_enhanced[test_mask]['semana_epi'].values,
        'reales': results['y_test_orig'],
        'predichos': results['final_test_pred'],
        'error': np.abs(results['y_test_orig'] - results['final_test_pred'])
    })
    pred_df.to_excel(writer, sheet_name='Predicciones', index=False)
    
    # Características importantes
    importance_df = pd.DataFrame({
        'Feature': selected_features,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    importance_df.to_excel(writer, sheet_name='Características', index=False)

print(f"Excel guardado en: {excel_file}")

# ==================== 12. RESULTADO FINAL ====================
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)
print(f"✓ Predictores originales: {len(predictor_cols)}")
print(f"✓ Predictores seleccionados: {len(selected_features)}")
print(f"✓ Train: {len(X_train)} registros")
print(f"✓ Test: {len(X_test)} registros")
print(f"\nTEST - MAE: {test_metrics['mae']:.2f}")
print(f"TEST - Peak MAE (80%): {test_metrics['peak_mae']:.2f}")
print(f"TEST - R²: {test_metrics['r2']:.4f}")

if test_metrics['mae'] < 7:
    print("\n✅ OBJETIVO ALCANZADO: MAE < 7")
else:
    print(f"\n⚠ Objetivo MAE < 7: actual {test_metrics['mae']:.2f}")

# Guardar modelo
model_file = os.path.join(output_dir, 'modelo_optimizado.txt')
results['models'][0].save_model(model_file)
print(f"Modelo guardado en: {model_file}")
print("="*70)

Cargando datos...
Predictores originales: 19

1. Limpiando datos...

2. Creando nuevas características (optimizado)...
Dimensiones después de feature engineering: (270, 38)
Nuevos predictores: 34

3. Aplicando transformación al target...

4. Seleccionando predictores más relevantes...
  Seleccionando con Random Forest...
Top 30 predictores seleccionados de 34

5. Preparando datos para entrenamiento...
Train: 249, Test: 21

ENTRENANDO MODELO ENSEMBLE...

  Entrenando modelo 1: LightGBM con pesos
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[480]	train's l1: 0.0296681	test's l1: 0.334169
  Entrenando modelo 2: LightGBM estándar
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[195]	train's l1: 0.0723134	test's l1: 0.33329
  Creando predicción ensemble (promedio ponderado)...

RESULTADOS DEL MODELO
Train MAE: 0.66 | R²: 0.9977
Test MAE: 2.87 | R²: 0.8448
Test Peak MAE (80%): 3.44

10. Gener